In [ ]:
# Import requirements
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

CSV_PATH = "landmarks_dataset_handedness_fixed.csv"
MODEL_PATH = "mediapipe_model_handedness_fixed.keras"
LABELS_PATH = "mediapipe_labels_handedness_fixed.json"

In [ ]:
# read the csv the had been generated by hand_landmark model which contains the coordinates for each hand in the dataset
df = pd.read_csv(CSV_PATH)

feature_cols = [col for col in df.columns if col.startswith(("x", "y", "z"))]
X = df[feature_cols].values.astype("float32")
y = df["label"].values
splits = df["split"].values

In [ ]:
# Apply the normalization function to every sample in X
# and store the result back as a NumPy array of type float32
# Scale all coordinates so the hand size becomes normalized
def normalize_landmarks(sample):
    sample = sample.reshape(21, 3).copy()

    wrist = sample[0].copy()
    sample = sample - wrist

    max_val = np.max(np.abs(sample))
    if max_val > 0:
        sample = sample / max_val

    return sample.flatten()

X = np.array([normalize_landmarks(sample) for sample in X], dtype=np.float32)


In [ ]:
# encode the dataset (csv) because it contains some categorical values such as label,split,...
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(label_encoder.classes_.tolist(), f)

In [ ]:
# read each split separately 
X_train = X[splits == "train"]
y_train = y_encoded[splits == "train"]

X_val = X[splits == "val"]
y_val = y_encoded[splits == "val"]

X_test = X[splits == "test"]
y_test = y_encoded[splits == "test"]

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Classes:", label_encoder.classes_.tolist())

Train: (104754, 63) (104754,)
Val: (22434, 63) (22434,)
Test: (22447, 63) (22447,)
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
# build the model with input layer 3 dense and dropout and output layer
# input -> dense(relu) -> dropout(0.3) -> dense(relu) -> dropout(0.3) -> dense(relu) -> output(softmax)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(63,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation="softmax")
])

In [ ]:
# compile the model with adam optimizer and lr = 0.001 
# loss = sparse_categorical_crossentropy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# initialize callback to prevent overfitting
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        verbose=1
    )
]

In [ ]:
# run the model with 100 epochs and batch size 32
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
3242/3274 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7145 - loss: 0.9090
Epoch 1: val_accuracy improved from None to 0.99688, saving model to mediapipe_sad_model_handedness_fixed.keras
3274/3274 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.8865 - loss: 0.3621 - val_accuracy: 0.9969 - val_loss: 0.0177 - learning_rate: 0.0010
Epoch 2/100
3246/3274 ━━━━━━━━━━━━━━━━━━━━ 0s 974us/step - accuracy: 0.9867 - loss: 0.0509
Epoch 2: val_accuracy improved from 0.99688 to 0.99755, saving model to mediapipe_sad_model_handedness_fixed.keras
3274/3274 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9889 - loss: 0.0431 - val_accuracy: 0.9975 - val_loss: 0.0122 - learning_rate: 0.0010
Epoch 3/100
3271/3274 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9921 - loss: 0.0311  
Epoch 3: val_accuracy did not improve from 0.99755
3274/3274 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9925 - loss: 0.0296 - val_accuracy: 0.9961 - val_loss: 0.0145 - learning_rate: 0.0010
Epoch 4/100
3249/3274 

In [ ]:
# performance on the test data shows no overfitting and with perfect accuracy
best_model = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_.tolist()))


Test Loss: 0.0028
Test Accuracy: 0.9995

Confusion Matrix:
[[900   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0 900   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0 899   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0 855   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0   0 900   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0   0   0 900   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0 900   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0 900   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0 900   0   0   0   0   0   0   0   